In [3]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

In [4]:
from google.colab import userdata

ngrok_authtoken = "3ClDmzH22sqWEbRSdze4yo4Zgpf_7ycL9z7EcZyfCYuL92PAR"
line_channel_access_token = "1OQmRLQRLLrqhossKfNHRNPO7oKJBICehSGz0ZmTf/I3JuDjHP2cYZX9L8u/MRrHuO6MmNe+szkjDQSHLoKreAvJxy/k7n0QhuWs/M31wA6meIcAA7nMmcPUsi9VR2SWtKdL3ny/TSnTrMKfBml3CwdB04t89/1O/w1cDnyilFU="
line_channel_secret = "79de232aeab6cf1e0df8c7723791c4bc"
gemini_api_key = "AQ.Ab8RN6Ksrrnv6rlygbfbLfHhoOKDG47PtvcSLu6ZgE82EvKaPg"

port = 5051

In [5]:
from pyngrok import ngrok
import requests

ngrok.kill()
ngrok.set_auth_token(ngrok_authtoken)

tunnel = ngrok.connect(port)
webhook_url = tunnel.public_url

print("Webhook URL:", webhook_url)


def update_line_webhook(webhook_url):
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {"endpoint": webhook_url}

    res = requests.put(url, headers=headers, json=data)
    print("status:", res.status_code)
    print("response:", res.text)


update_line_webhook(webhook_url)

Webhook URL: https://gap-unison-monotone.ngrok-free.dev
status: 200
response: {}


In [6]:
from google import genai

client = genai.Client(api_key=gemini_api_key)


def stateless_query(payload):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=payload
    )
    return response.text

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import WebhookHandler
from linebot.v3.exceptions import InvalidSignatureError
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=["POST"])
def callback():

    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)

    print("BODY:", body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        abort(400)

    return "OK"


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):

    text = event.message.text

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # ====== 0701 重點註解 ======
        # 為何要用 "AI " 開頭：
        # 1. 用 startswith('AI ') 判斷是否要使用 Gemini
        # 2. AI 開頭才會把後面文字當 prompt 丟給 Gemini
        # 3. 沒有 AI 開頭就走一般回覆

        if text.startswith('AI '):

            prompt = text[3:]
            reply_text = stateless_query(prompt)

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[
                        TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)
                    ]
                )
            )


app.run(host="0.0.0.0", port=5051)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5051
 * Running on http://172.28.0.12:5051
INFO:werkzeug:Press CTRL+C to quit


BODY: {"destination":"U396699aed832fbe4626934d45e324849","events":[{"type":"message","message":{"type":"text","id":"616653024651641145","quoteToken":"aZv-fbeRnEdJvvcnRbdhyvh_Kb8hx3WEqow7Q8ivk0ss88yMGjvcZYCmf6OtzR5fbFMKC2bI5eUQr6fLnImSSA1tvBzKdygXAPNbroTDdUObIpMJFlprElPPlg2qfyZQbI5sa3sSNpHStOJBCIbKsQ","markAsReadToken":"BhWI-zq66fvbCE3LjE_3JMHZPIsDOxFP0H9iG5Gw04pNSpkFy04N0PDI9bsn5M9-c3NEsc_wGSRnTlAsD7g2HdjF7sCluhsomSJJVEdPG2YjwOdTxKYAaVxzjBQcy654VgBCKmtX8vnbTBmhx1f2UnQv8fypB4OZAba31bP_VoRY07sU_Y4Kb8bSgjmcK-dpHPjN4n3ccLPdfeCeuNUm4w","text":"AI簡介明新科技大學"},"webhookEventId":"01KT3KJWETDZSKDNYMHC188A4K","deliveryContext":{"isRedelivery":false},"timestamp":1780385149011,"source":{"type":"user","userId":"U440099af22104a3db5953a8118a10fe5"},"replyToken":"ef1df88e9cbc47e8b53a3d5ead9d64f3","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 07:25:50] "POST / HTTP/1.1" 200 -


BODY: {"destination":"U396699aed832fbe4626934d45e324849","events":[{"type":"message","message":{"type":"text","id":"616653228125716481","quoteToken":"_fFU6IAgKNwZtfm8yIqQbjvvnt6Lm9OFpVzZd8V5Z_PBOSUI3HjNyLy3AlPfA65JwdDPCRd-gQrxwQ1Wyupsmu4uJrIy7pVTvmbrXnlMrEHf4zAhSqAZSiCHPV56QHNgIA28i6mf4lpFpKg7tyw4eg","markAsReadToken":"fYjy_X7qykUTaUTjx7QZWXbZtgJJhJ7TWWz0GIprjVgJKRumDUitwT-ACrtMqIEJIhkXLR_5afoN7w7-L1QLwKkRvFNP5zD-aAYIELTTkmAeeUSl7fsWTSioOi4LUj3knuFOvD4_cC7yMH8SmafC2lz7f8_N2CzmiTTtBYzKuAzqeJyL-In0FTGIfAa_E18WQMwV9OWH9WLvh1nUwsxoaQ","text":"AI 簡介明新科技大學"},"webhookEventId":"01KT3KPJBXPCRH05W2RMQTV83V","deliveryContext":{"isRedelivery":false},"timestamp":1780385270126,"source":{"type":"user","userId":"U440099af22104a3db5953a8118a10fe5"},"replyToken":"c86c34bdc96a4168a3f964bfa2f32b8e","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 07:28:02] "POST / HTTP/1.1" 200 -
